# FinCompass - Banking Complaint Intelligence & EDA
This notebook performs Exploratory Data Analysis (EDA) on the synthesized RBI banking complaint dataset. It loads data from the SQLite database, performs descriptive statistics, visualizes the distribution of categories, analyzes YoY growth, and examines resolution times.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plotting style
sns.set_theme(style="whitegrid")

## 1. Connect to Database & Ingest Data

In [ ]:
DB_PATH = Path("../database/fincompass.db")
conn = sqlite3.connect(str(DB_PATH))

# Load full complaints joined table
query = """
    SELECT 
        c.complaint_id, c.date, c.complaint_text, c.state, c.channel, c.status,
        c.resolution_days, c.customer_segment, c.year, c.month, c.quarter,
        b.bank_name, b.bank_type, cat.category_name, cat.subcategory_name
    FROM complaints c
    JOIN banks b ON c.bank_id = b.bank_id
    JOIN categories cat ON c.subcategory_id = cat.subcategory_id
"""
df = pd.read_sql_query(query, conn)
df["date"] = pd.to_datetime(df["date"])
df.head()

## 2. Descriptive Summary

In [ ]:
print(f"Total complaints: {len(df)}")
print(f"Resolution status distribution:\n{df['status'].value_counts(normalize=True) * 100}")
print(f"\nAverage resolution days (overall): {df['resolution_days'].mean():.2f} days")

## 3. Bank-wise Complaint Volumes

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=df, y="bank_name", order=df["bank_name"].value_counts().index, palette="Blues_r")
plt.title("Total Complaint Volume by Bank")
plt.xlabel("Complaint Count")
plt.ylabel("Bank")
plt.tight_layout()
plt.show()

## 4. Resolution Velocity Analysis (Public vs Private)

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df[df["status"] == "Resolved"], x="bank_type", y="resolution_days", palette="Set2")
plt.title("Complaint Resolution Days by Bank Type")
plt.xlabel("Bank Type")
plt.ylabel("Resolution Days")
plt.show()

## 5. Temporal Trend of Digital Banking Fraud

In [ ]:
fraud_df = df[df["category_name"] == "Digital Banking Fraud"].copy()
fraud_monthly = fraud_df.groupby(["year", "month"]).size().reset_index(name="count")
fraud_monthly["date"] = pd.to_datetime(fraud_monthly["year"].astype(str) + "-" + fraud_monthly["month"].astype(str) + "-01")

plt.figure(figsize=(12, 5))
sns.lineplot(data=fraud_monthly, x="date", y="count", marker="o", color="darkred")
plt.title("Rising Trend of Digital Banking Fraud (Monthly Count)")
plt.xlabel("Date")
plt.ylabel("Complaint Count")
plt.show()

In [ ]:
conn.close()